<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 15 · RL Algorithms for Asset Management

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals

This notebook turns Chapter 15 into runnable RL experiments:

- define a toy single-asset rebalancing environment, including states, actions, and rewards,
- train a small tabular Q-learning agent and inspect its episodic rewards, and
- sketch a simple policy-gradient update to highlight similarities and differences.

Everything runs quickly on CPU and is meant for intuition rather than production training.

### Getting Help While Implementing RL
- **Chapter 14** explains states/actions/rewards reused here.
- **Appendix A** summarizes policy-gradient math.
- **Chapter 8** reminds you how to evaluate strategies.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

from dataclasses import dataclass

### Data Loading
We load the price panel once for this notebook.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date").sort_index().ffill()

## 1. Environment Setup
We reuse the Chapter 14 environment with slight tweaks for learning.

In [ ]:
@dataclass
class SimpleEnv:
    prices: pd.Series
    cost: float = 0.0005

    def reset(self):
        self.t = 0
        self.position = 0
        return self.state()

    def state(self):
        window = self.prices.iloc[max(0, self.t - 20): self.t]
        if len(window):
            momentum = window.pct_change().mean()
        else:
            momentum = 0.0
        if momentum is None or not np.isfinite(momentum):
            momentum = 0.0
        return np.array([float(momentum), float(self.position)])

    def step(self, action):
        price_now = self.prices.iloc[self.t]
        price_next = self.prices.iloc[self.t + 1]
        ret = price_next / price_now - 1
        reward = action * ret - self.cost * abs(action - self.position)
        self.position = action
        self.t += 1
        done = self.t >= len(self.prices) - 2
        return self.state(), reward, done

series = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date")["AAPL"].ffill()
env = SimpleEnv(series)
_ = env.reset()

## 2. Tabular Q-Learning

We now let the agent learn a Q‑table that maps coarse states to preferred actions through repeated interaction with the environment.

In [ ]:
actions = [-1, 0, 1]
q_values = {}
alpha = 0.1
gamma = 0.95
epsilon = 0.1


def discretize(state):
    clean = np.nan_to_num(state, nan=0.0)
    return tuple((clean * 50).astype(int))


n_episodes = 200
for episode in range(n_episodes):
    state = env.reset()
    done = False
    while not done:
        key = discretize(state)
        q_values.setdefault(key, np.zeros(len(actions)))
        if np.random.rand() < epsilon:
            idx = np.random.randint(len(actions))
        else:
            idx = np.argmax(q_values[key])
        action = actions[idx]
        next_state, reward, done = env.step(action)
        next_key = discretize(next_state)
        q_values.setdefault(next_key, np.zeros(len(actions)))
        target = reward + gamma * q_values[next_key].max()
        q_values[key][idx] += alpha * (target - q_values[key][idx])
        state = next_state
    if (episode + 1) % 50 == 0:
        print(f"Completed Q-learning episode {episode + 1}/{n_episodes}")

len(q_values)

### 2.1 Evaluate Learned Policy

With a Q‑table in hand, we simulate fresh episodes using greedy actions to see the average reward of the learned policy.

In [ ]:
def run_policy(policy_vals, episodes=20):
    rewards = []
    for _ in range(episodes):
        state = env.reset()
        done = False
        total = 0
        while not done:
            key = discretize(state)
            action = actions[np.argmax(policy_vals.get(key, np.zeros(len(actions))))]
            state, reward, done = env.step(action)
            total += reward
        rewards.append(total)
    return np.mean(rewards)

avg_reward = run_policy(q_values)
avg_reward

## 3. Policy-Gradient Sketch

Finally, we switch perspectives and implement a tiny policy‑gradient loop that optimizes action probabilities directly, contrasting it with Q‑learning.

In [ ]:
theta = np.zeros(2)
eta = 0.05

def softmax_policy(state):
    logits = np.array([np.dot(theta, [state[0], action]) for action in actions])
    exp_logits = np.exp(logits - logits.max())
    return exp_logits / exp_logits.sum()

for episode in range(120):
    state = env.reset()
    done = False
    states, actions_taken, rewards = [], [], []
    while not done:
        probs = softmax_policy(state)
        action = np.random.choice(actions, p=probs)
        states.append(state)
        actions_taken.append(action)
        state, reward, done = env.step(action)
        rewards.append(reward)
    G = sum(rewards)
    grad = np.zeros_like(theta)
    for s, a in zip(states, actions_taken):
        probs = softmax_policy(s)
        idx = actions.index(a)
        grad += (1 - probs[idx]) * np.array([s[0], a])
    theta += eta * G * grad

theta

## 4. Exercises

### Exercise 1 – Reward Normalization
Normalize rewards by recent volatility to reduce variance.
<details><summary>Hint</summary>
Track a rolling volatility estimate and divide rewards by it.
</details>

### Exercise 2 – Expanded Action Set
Allow half-sized positions (±0.5) in addition to full ones.
<details><summary>Hint</summary>
Extend the <code>actions</code> list to include ±0.5 and update Q-value tables.
</details>

### Exercise 3 – Policy Baseline
Add a moving-average baseline to the policy-gradient update.
<details><summary>Hint</summary>
Maintain an exponential moving average of episode returns and subtract it from <code>G</code>.
</details>


## 5. Takeaways
- Q-learning and policy gradients respond differently to sparse rewards.
- Reward shaping and transaction costs matter for stability.
- RL outputs ultimately feed into familiar portfolio metrics.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">